# 🛢️ Oil Spill Detection — Google Colab + Google Drive
**GAN-augmented dataset → DeepLabV3+ / EfficientNet-B4 segmentation**

Improvements over baseline:
- **EfficientNet-B4** encoder (stronger than B3)
- **512×512** input resolution
- **Focal + Dice loss** (handles class imbalance better)
- **Label smoothing** on CE component
- **OneCycleLR** scheduler (faster convergence)
- **Stronger augmentations** (GridDistortion, ElasticTransform, CoarseDropout)
- **CRF post-processing** option
- Proper **per-class metrics** printed each epoch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q segmentation-models-pytorch albumentations lpips pytorch-msssim reportlab torchinfo scipy

In [ ]:
import os, cv2, json, copy, time, random, shutil, warnings, datetime, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage

from tqdm import tqdm
from glob import glob
from pathlib import Path
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import lpips
from pytorch_msssim import ssim
import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print('Device :', DEVICE)
print('PyTorch:', torch.__version__)
print('=' * 60)

In [ ]:
# ── Google Drive Paths ────────────────────────────────────────────────
BASE = '/content/drive/MyDrive/oilspill'   # ← edit if needed

TRAIN_IMG  = os.path.join(BASE, 'train/images')
TRAIN_MASK = os.path.join(BASE, 'train/masks')
TEST_IMG   = os.path.join(BASE, 'test/images')
TEST_MASK  = os.path.join(BASE, 'test/masks')

WORK_DIR   = '/content/drive/MyDrive/oilspill/outputs'
MODEL_DIR  = os.path.join(WORK_DIR, 'models')
RESULT_DIR = os.path.join(WORK_DIR, 'results')
GAN_SAVE   = MODEL_DIR

for d in [MODEL_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Train images:', len(os.listdir(TRAIN_IMG)))
print('Train masks :', len(os.listdir(TRAIN_MASK)))
print('Test  images:', len(os.listdir(TEST_IMG)))
print('Test  masks :', len(os.listdir(TEST_MASK)))

In [ ]:
def get_valid_stems(img_dir, mask_dir):
    def stem_map(d):
        m = {}
        for f in os.listdir(d):
            n, e = os.path.splitext(f)
            if e.lower() in ('.png', '.jpg', '.jpeg'):
                m[n] = f
        return m
    imgs  = stem_map(img_dir)
    masks = stem_map(mask_dir)
    common = sorted(imgs.keys() & masks.keys())
    print(f'Images: {len(imgs)}  Masks: {len(masks)}  Pairs: {len(common)}')
    return common

def detect_corrupt(stems, img_dir, mask_dir):
    bad = []
    for stem in tqdm(stems, desc='Checking files'):
        def find(d):
            for e in ['.png','.jpg','.jpeg']:
                p = os.path.join(d, stem+e)
                if os.path.exists(p): return p
        ip, mp = find(img_dir), find(mask_dir)
        if ip is None or mp is None:
            bad.append(stem); continue
        if cv2.imread(ip) is None or cv2.imread(mp) is None:
            bad.append(stem)
    return bad

COMMON_FILES = get_valid_stems(TRAIN_IMG, TRAIN_MASK)
BAD_FILES    = detect_corrupt(COMMON_FILES, TRAIN_IMG, TRAIN_MASK)
COMMON_FILES = [f for f in COMMON_FILES if f not in BAD_FILES]
print(f'\nValid: {len(COMMON_FILES)}  Corrupt: {len(BAD_FILES)}')

In [ ]:
NUM_CLASSES  = 4
CLASS_NAMES  = ['Background', 'Oil', 'Water', 'Other']
CLASS_COLORS = {
    0: (0,   0,   0),
    1: (255, 0,   124),
    2: (51,  221, 255),
    3: (255, 204, 51),
}
COLOR_TO_CLASS = {v: k for k, v in CLASS_COLORS.items()}

def rgb_to_mask(mask_rgb):
    rgb_int = (mask_rgb[:,:,0].astype(np.uint32) << 16 |
               mask_rgb[:,:,1].astype(np.uint32) <<  8 |
               mask_rgb[:,:,2].astype(np.uint32))
    lut  = {(r<<16)|(g<<8)|b: cls for (r,g,b),cls in COLOR_TO_CLASS.items()}
    mask = np.zeros(mask_rgb.shape[:2], dtype=np.uint8)
    for key, val in lut.items():
        mask[rgb_int == key] = val
    return mask

def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for idx, color in CLASS_COLORS.items():
        rgb[mask == idx] = color
    return rgb

In [ ]:
# ── IMPROVEMENT: 512×512 resolution + stronger augmentation ──────────
IMG_SIZE = 512

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                       rotate_limit=20, border_mode=cv2.BORDER_REFLECT_101, p=0.6),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.ElasticTransform(alpha=80, sigma=10, p=1.0),
        A.OpticalDistortion(distort_limit=0.2, p=1.0),
    ], p=0.4),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
    A.HueSaturationValue(15, 25, 25, p=0.4),
    A.CLAHE(clip_limit=4, tile_grid_size=(8,8), p=0.3),
    A.GaussianBlur(blur_limit=5, p=0.25),
    A.GaussNoise(var_limit=(10,50), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    fill_value=0, p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

In [ ]:
class OilDataset(Dataset):
    def __init__(self, image_dir, mask_dir, files, transform=None):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.files     = files
        self.transform = transform

    def __len__(self): return len(self.files)

    def _find_file(self, directory, stem):
        for ext in ['.png','.jpg','.jpeg','.PNG','.JPG','.JPEG']:
            p = os.path.join(directory, stem+ext)
            if os.path.exists(p): return p
        stem_lower = stem.lower()
        try:
            for fname in os.listdir(directory):
                n, e = os.path.splitext(fname)
                if n.lower() == stem_lower and e.lower() in ('.png','.jpg','.jpeg'):
                    return os.path.join(directory, fname)
        except OSError: pass
        return None

    def __getitem__(self, idx):
        stem = self.files[idx]
        ip   = self._find_file(self.image_dir, stem)
        mp   = self._find_file(self.mask_dir,  stem)
        if ip is None: raise FileNotFoundError(f'No image for {stem}')
        if mp is None: raise FileNotFoundError(f'No mask  for {stem}')
        img  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(cv2.imread(mp), cv2.COLOR_BGR2RGB)
        mask = rgb_to_mask(mask)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        if img.dim() == 4:  img  = img.squeeze(0)
        if mask.dim() == 3: mask = mask.squeeze(0)
        return img, torch.as_tensor(mask, dtype=torch.long)

In [ ]:
all_files = get_valid_stems(TRAIN_IMG, TRAIN_MASK)
all_files = [f for f in all_files if f not in BAD_FILES]

train_files, val_files = train_test_split(
    all_files, test_size=0.15, random_state=42, shuffle=True)
print(f'Train: {len(train_files)}  Val: {len(val_files)}')

train_ds = OilDataset(TRAIN_IMG, TRAIN_MASK, train_files, train_transform)
val_ds   = OilDataset(TRAIN_IMG, TRAIN_MASK, val_files,   valid_transform)

BATCH_SIZE = 4  # reduce if OOM; increase to 8 with A100

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for i in range(3):
    stem = random.choice(COMMON_FILES)
    def fp(d, s):
        for e in ['.png','.jpg','.jpeg']:
            p = os.path.join(d, s+e)
            if os.path.exists(p): return p
    ip, mp = fp(TRAIN_IMG, stem), fp(TRAIN_MASK, stem)
    if ip is None or mp is None: continue
    img  = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    mask = cv2.cvtColor(cv2.imread(mp), cv2.COLOR_BGR2RGB)
    midx = rgb_to_mask(mask)
    axes[i,0].imshow(img);         axes[i,0].set_title('Image');      axes[i,0].axis('off')
    axes[i,1].imshow(mask);        axes[i,1].set_title('Mask (RGB)'); axes[i,1].axis('off')
    axes[i,2].imshow(midx,cmap='tab10',vmin=0,vmax=3)
    axes[i,2].set_title('Mask (Index)'); axes[i,2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def init_weights(model):
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.InstanceNorm2d):
            if m.weight is not None: nn.init.ones_(m.weight)
            if m.bias   is not None: nn.init.zeros_(m.bias)

class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch))
    def forward(self,x): return x + self.block(x)

class DownBlock(nn.Module):
    def __init__(self, ic, oc, norm=True):
        super().__init__()
        l = [nn.Conv2d(ic,oc,4,2,1,bias=False)]
        if norm: l.append(nn.InstanceNorm2d(oc))
        l.append(nn.LeakyReLU(0.2,True))
        self.block = nn.Sequential(*l)
    def forward(self,x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, ic, oc, drop=False):
        super().__init__()
        l = [nn.ConvTranspose2d(ic,oc,4,2,1,bias=False),
             nn.InstanceNorm2d(oc), nn.ReLU(True)]
        if drop: l.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*l)
    def forward(self,x): return self.block(x)

class Generator(nn.Module):
    def __init__(self, ic=1, oc=3, f=64):
        super().__init__()
        self.d1 = DownBlock(ic,  f,   norm=False)
        self.d2 = DownBlock(f,   f*2)
        self.d3 = DownBlock(f*2, f*4)
        self.d4 = DownBlock(f*4, f*8)
        self.bn = nn.Sequential(nn.Conv2d(f*8,f*8,4,2,1), nn.ReLU(True))
        self.u1 = UpBlock(f*8,  f*8, drop=True)
        self.u2 = UpBlock(f*16, f*4)
        self.u3 = UpBlock(f*8,  f*2)
        self.u4 = UpBlock(f*4,  f)
        self.fin= nn.Sequential(nn.ConvTranspose2d(f*2,oc,4,2,1), nn.Tanh())
    def forward(self,x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2); d4=self.d4(d3)
        b=self.bn(d4)
        u1=self.u1(b);  u2=self.u2(torch.cat([u1,d4],1))
        u3=self.u3(torch.cat([u2,d3],1)); u4=self.u4(torch.cat([u3,d2],1))
        return self.fin(torch.cat([u4,d1],1))

class Discriminator(nn.Module):
    def __init__(self, ic=4, f=64):
        super().__init__()
        self.model = nn.Sequential(
            DownBlock(ic,f,norm=False), DownBlock(f,f*2), DownBlock(f*2,f*4),
            nn.Conv2d(f*4,f*8,4,1,1), nn.InstanceNorm2d(f*8),
            nn.LeakyReLU(0.2,True), nn.Conv2d(f*8,1,4,1,1))
    def forward(self,mask,img): return self.model(torch.cat([mask,img],1))

gen  = Generator(ic=1, oc=3, f=64).to(DEVICE); init_weights(gen)
disc = Discriminator(ic=4, f=64).to(DEVICE);    init_weights(disc)
print('Generator params :', sum(p.numel() for p in gen.parameters()))
print('Discriminator params:', sum(p.numel() for p in disc.parameters()))

In [ ]:
GAN_EPOCHS   = 50
LR           = 2e-4
LAMBDA_L1    = 50
LAMBDA_LPIPS = 10
LAMBDA_SSIM  = 5

opt_gen  = torch.optim.Adam(gen.parameters(),  lr=LR, betas=(0.5,0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=LR, betas=(0.5,0.999))
sched_gen  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gen,  T_max=GAN_EPOCHS)
sched_disc = torch.optim.lr_scheduler.CosineAnnealingLR(opt_disc, T_max=GAN_EPOCHS)

bce        = nn.BCEWithLogitsLoss()
l1_loss    = nn.L1Loss()
lpips_loss = lpips.LPIPS(net='alex').to(DEVICE)
scaler_g   = GradScaler(); scaler_d = GradScaler()

ema_decay = 0.999
ema_gen   = copy.deepcopy(gen); ema_gen.eval()
for p in ema_gen.parameters(): p.requires_grad_(False)

@torch.no_grad()
def update_ema(m, em, decay=0.999):
    sd = m.state_dict()
    for k,v in em.state_dict().items():
        if k in sd: v.copy_(decay*v + (1-decay)*sd[k])

gan_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

best_g = 1e9; gan_history = {'g':[],'d':[]}

resume = False
if resume:
    ckpt = torch.load(f'{GAN_SAVE}/gan_checkpoint.pth', map_location=DEVICE)
    gen.load_state_dict(ckpt['gen']); disc.load_state_dict(ckpt['disc'])
    opt_gen.load_state_dict(ckpt['opt_gen']); opt_disc.load_state_dict(ckpt['opt_disc'])
    ema_gen.load_state_dict(ckpt['ema']); best_g = ckpt['best_g']
    print('Resumed from checkpoint')

for epoch in range(GAN_EPOCHS):
    gen.train(); disc.train()
    g_ep = d_ep = 0.0
    for real, mask in gan_loader:
        mask_f = mask.to(DEVICE).float().unsqueeze(1)
        real   = real.to(DEVICE)
        with autocast():
            fake   = gen(mask_f)
            d_real = disc(mask_f, real)
            d_fake = disc(mask_f, fake.detach())
            loss_d = 0.5*(bce(d_real, torch.full_like(d_real,0.9)) +
                          bce(d_fake, torch.full_like(d_fake,0.1)))
        opt_disc.zero_grad(set_to_none=True)
        scaler_d.scale(loss_d).backward()
        scaler_d.unscale_(opt_disc)
        torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
        scaler_d.step(opt_disc); scaler_d.update()
        with autocast():
            fake     = gen(mask_f)
            gan_loss = bce(disc(mask_f,fake), torch.ones_like(disc(mask_f,fake)))
            l1       = l1_loss(fake, real)
            lp       = lpips_loss(fake, real).mean()
            ssim_v   = 1 - ssim((fake+1)/2,(real+1)/2,data_range=1.0,size_average=True)
            loss_g   = gan_loss + LAMBDA_L1*l1 + LAMBDA_LPIPS*lp + LAMBDA_SSIM*ssim_v
        opt_gen.zero_grad(set_to_none=True)
        scaler_g.scale(loss_g).backward()
        scaler_g.unscale_(opt_gen)
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        scaler_g.step(opt_gen); scaler_g.update()
        update_ema(gen, ema_gen)
        g_ep += loss_g.item(); d_ep += loss_d.item()
    sched_gen.step(); sched_disc.step()
    g_avg = g_ep/len(gan_loader); d_avg = d_ep/len(gan_loader)
    print(f'GAN Epoch {epoch+1:03d}/{GAN_EPOCHS} | G: {g_avg:.4f} | D: {d_avg:.4f}')
    if g_avg < best_g:
        best_g = g_avg
        torch.save(ema_gen.state_dict(), f'{GAN_SAVE}/best_generator.pth')
        print('  ✅ Best generator saved')
    torch.save({'gen':gen.state_dict(),'disc':disc.state_dict(),
                'opt_gen':opt_gen.state_dict(),'opt_disc':opt_disc.state_dict(),
                'ema':ema_gen.state_dict(),'best_g':best_g,
                'epoch':epoch}, f'{GAN_SAVE}/gan_checkpoint.pth')
print('✅ GAN Training Complete')

In [ ]:
# ── IMPROVEMENT: Focal Loss + Dice Loss (better for imbalanced classes) ─

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, pred, target):
        # label smoothing on the one-hot target
        n_cls = pred.size(1)
        oh    = F.one_hot(target, n_cls).permute(0,3,1,2).float()
        smooth_oh = oh * (1 - self.label_smoothing) + self.label_smoothing / n_cls
        log_p = F.log_softmax(pred, dim=1)
        p     = torch.exp(log_p)
        focal_w = (1 - p) ** self.gamma
        if self.weight is not None:
            w = self.weight.view(1,-1,1,1)
            focal_w = focal_w * w
        loss = -(focal_w * smooth_oh * log_p).sum(dim=1).mean()
        return loss


class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.softmax(pred, dim=1)
        target = F.one_hot(target, NUM_CLASSES).permute(0,3,1,2).float()
        inter  = (pred * target).sum((2,3))
        union  = pred.sum((2,3)) + target.sum((2,3))
        dice   = (2*inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


dice_fn  = DiceLoss()
focal_fn = FocalLoss(gamma=2.0, label_smoothing=0.05)

def segmentation_loss(pred, target, class_weights=None):
    if class_weights is not None:
        focal_fn.weight = class_weights
    return focal_fn(pred, target) + 0.5 * dice_fn(pred, target)

print('Loss functions ready.')

In [ ]:
# ── IMPROVEMENT: EfficientNet-B4 encoder (larger capacity than B3) ────
seg_model = smp.DeepLabV3Plus(
    encoder_name    = 'efficientnet-b4',
    encoder_weights = 'imagenet',
    in_channels     = 3,
    classes         = NUM_CLASSES,
    activation      = None,
    decoder_atrous_rates = (6, 12, 24),   # larger receptive field
).to(DEVICE)

scaler = GradScaler()
print('Segmentation model (DeepLabV3+ / EfficientNet-B4) ready.')

In [ ]:
def pixel_accuracy(pred, target):
    pred = torch.argmax(pred, dim=1)
    return (pred == target).float().mean().item()

def mean_iou(pred, target, n_classes):
    pred = torch.argmax(pred, dim=1)
    ious = []
    for cls in range(n_classes):
        inter = ((pred==cls)&(target==cls)).sum().item()
        union = ((pred==cls)|(target==cls)).sum().item()
        if union > 0: ious.append(inter/union)
    return float(np.mean(ious)) if ious else 0.0

def per_class_iou(pred, target, n_classes):
    pred = torch.argmax(pred, dim=1)
    ious = {}
    for cls in range(n_classes):
        inter = ((pred==cls)&(target==cls)).sum().item()
        union = ((pred==cls)|(target==cls)).sum().item()
        ious[CLASS_NAMES[cls]] = inter/union if union > 0 else float('nan')
    return ious

def dice_score(pred, target):
    pred   = torch.argmax(pred, dim=1).flatten()
    target = target.flatten()
    inter  = (pred==target).sum().item()
    return (2*inter)/(len(pred)+len(target))

In [ ]:
SYNTH_ROOT = '/content/synthetic_dataset'
SYNTH_IMG  = os.path.join(SYNTH_ROOT,'images')
SYNTH_MASK = os.path.join(SYNTH_ROOT,'masks')
os.makedirs(SYNTH_IMG, exist_ok=True); os.makedirs(SYNTH_MASK, exist_ok=True)

gen.load_state_dict(torch.load(f'{GAN_SAVE}/best_generator.pth', map_location=DEVICE))
gen.eval(); print('✅ Best Generator Loaded')

NUM_SYNTHETIC = 1000; generated = 0

while generated < NUM_SYNTHETIC:
    for real, mask in tqdm(gan_loader, leave=False):
        mask_f = mask.to(DEVICE).float().unsqueeze(1)
        with torch.no_grad():
            fake = gen(mask_f).cpu()
        for b in range(fake.size(0)):
            img_np = np.clip((fake[b].permute(1,2,0).numpy()+1)/2,0,1)
            img_np = (img_np*255).astype(np.uint8)
            generated += 1
            cv2.imwrite(os.path.join(SYNTH_IMG,  f'synth_{generated:05d}.png'),
                        cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
            mask_rgb = mask_to_rgb(mask[b].numpy().astype(np.uint8))
            cv2.imwrite(os.path.join(SYNTH_MASK, f'synth_{generated:05d}.png'),
                        cv2.cvtColor(mask_rgb, cv2.COLOR_RGB2BGR))
            if generated >= NUM_SYNTHETIC: break
        if generated >= NUM_SYNTHETIC: break

print(f'Generated: {generated} synthetic pairs')

In [ ]:
MERGED_ROOT = '/content/merged_dataset'
MERGED_IMG  = os.path.join(MERGED_ROOT,'images')
MERGED_MASK = os.path.join(MERGED_ROOT,'masks')
os.makedirs(MERGED_IMG, exist_ok=True); os.makedirs(MERGED_MASK, exist_ok=True)

def find_file(d, stem):
    for e in ['.png','.jpg','.jpeg','.PNG','.JPG','.JPEG']:
        p = os.path.join(d, stem+e)
        if os.path.exists(p): return p

real_stems = [s for s in get_valid_stems(TRAIN_IMG, TRAIN_MASK) if s not in BAD_FILES]
count = 0
for stem in real_stems:
    si, sm = find_file(TRAIN_IMG,stem), find_file(TRAIN_MASK,stem)
    if si and sm:
        shutil.copy2(si, os.path.join(MERGED_IMG,  f'real_{count:05d}.png'))
        shutil.copy2(sm, os.path.join(MERGED_MASK, f'real_{count:05d}.png'))
        count += 1
print(f'Real pairs copied: {count}')

si_files = sorted(os.listdir(SYNTH_IMG)); sm_files = sorted(os.listdir(SYNTH_MASK))
for i,(sf,mf) in enumerate(zip(si_files, sm_files)):
    shutil.copy2(os.path.join(SYNTH_IMG, sf),  os.path.join(MERGED_IMG,  f'synth_{i:05d}.png'))
    shutil.copy2(os.path.join(SYNTH_MASK, mf), os.path.join(MERGED_MASK, f'synth_{i:05d}.png'))
print(f'Synth pairs copied: {i+1}')

all_merged  = get_valid_stems(MERGED_IMG, MERGED_MASK)
train_files, val_files = train_test_split(all_merged,test_size=0.15,shuffle=True,random_state=42)
print(f'Training: {len(train_files)}  Validation: {len(val_files)}')

train_dataset = OilDataset(MERGED_IMG, MERGED_MASK, train_files, train_transform)
val_dataset   = OilDataset(MERGED_IMG, MERGED_MASK, val_files,   valid_transform)

In [ ]:
# Class pixel counts → inverse-frequency weights
counts = np.zeros(NUM_CLASSES)
for _, mask in train_dataset:
    m = mask.numpy()
    for c in range(NUM_CLASSES): counts[c] += np.sum(m==c)
weights   = counts.sum()/(NUM_CLASSES * counts + 1e-8)
weights_t = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print('Class weights:', weights_t.cpu().numpy().round(4))

# Oil-pixel weighted sampler
sample_weights = []
for stem in train_files:
    mp = find_file(MERGED_MASK, stem)
    if mp is None: sample_weights.append(1.0); continue
    mbgr = cv2.imread(mp)
    if mbgr is None: sample_weights.append(1.0); continue
    midx = rgb_to_mask(cv2.cvtColor(mbgr, cv2.COLOR_BGR2RGB))
    oil_ratio = np.sum(midx==1)/midx.size
    sample_weights.append(1.0 + oil_ratio*20)

sampler = WeightedRandomSampler(
    torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader = valid_loader = DataLoader(val_dataset, batch_size=4, shuffle=False,
                                       num_workers=2, pin_memory=True)
print('✅ Weighted DataLoaders ready')

In [ ]:
# ── IMPROVEMENT: OneCycleLR + differential LR + focal+dice loss ─────────

SEG_EPOCHS = 60
ACCUM      = 4

optimizer = torch.optim.AdamW([
    {'params': seg_model.encoder.parameters(),           'lr': 5e-6},
    {'params': seg_model.decoder.parameters(),           'lr': 2e-4},
    {'params': seg_model.segmentation_head.parameters(), 'lr': 4e-4},
], weight_decay=1e-4)

# OneCycleLR — typically converges 10-20% faster than CosineAnnealingWarmRestarts
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr   = [5e-5, 2e-3, 4e-3],
    steps_per_epoch = len(train_loader),
    epochs   = SEG_EPOCHS,
    pct_start= 0.1,
    div_factor = 10,
    final_div_factor = 1e4,
)

history = {k: [] for k in
    ['train_loss','val_loss','iou','dice','oil_iou','oil_f1','lr']}

best_iou = 0.0; best_score = 0.0
patience = 15;  counter    = 0
hard_examples = []

for epoch in range(SEG_EPOCHS):

    if epoch == 1:
        for m in seg_model.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()

    # ── Train ──
    seg_model.train(); train_loss = 0.0; optimizer.zero_grad()
    for i, (img, mask) in enumerate(train_loader):
        img  = img.to(DEVICE); mask = mask.to(DEVICE)
        with autocast():
            pred = seg_model(img)
            loss = segmentation_loss(pred, mask, weights_t) / ACCUM
        scaler.scale(loss).backward()
        if (i+1)%ACCUM==0 or (i+1)==len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(seg_model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
        batch_loss = loss.item()*ACCUM
        train_loss += batch_loss
        if batch_loss > 0.5: hard_examples.append(batch_loss)
        scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate ──
    seg_model.eval()
    vl = vi = vd = oil_iou = oil_f1 = 0.0
    with torch.no_grad():
        for img, mask in valid_loader:
            img  = img.to(DEVICE); mask = mask.to(DEVICE)
            pred = seg_model(img)
            vl  += segmentation_loss(pred, mask, weights_t).item()
            vi  += mean_iou(pred, mask, NUM_CLASSES)
            vd  += dice_score(pred, mask)
            # per-class Oil IoU
            pc   = per_class_iou(pred, mask, NUM_CLASSES)
            oi   = pc.get('Oil', 0.0)
            if not np.isnan(oi): oil_iou += oi
            # Oil F1
            p2 = torch.argmax(pred,1).flatten().cpu().numpy()
            g2 = mask.flatten().cpu().numpy()
            eps2 = 1e-7
            op_n = np.sum((p2==1)&(g2==1))
            prec = op_n/(np.sum(p2==1)+eps2)
            rec  = op_n/(np.sum(g2==1)+eps2)
            oil_f1 += 2*prec*rec/(prec+rec+eps2)
    n = len(valid_loader)
    vl/=n; vi/=n; vd/=n; oil_iou/=n; oil_f1/=n

    history['train_loss'].append(train_loss)
    history['val_loss'].append(vl)
    history['iou'].append(vi)
    history['dice'].append(vd)
    history['oil_iou'].append(oil_iou)
    history['oil_f1'].append(oil_f1)
    history['lr'].append(optimizer.param_groups[1]['lr'])

    score = 0.5*vi + 0.3*oil_iou + 0.2*vd

    print(f'Epoch {epoch+1:03d}/{SEG_EPOCHS}'
          f' | TrL:{train_loss:.4f} | VlL:{vl:.4f}'
          f' | mIoU:{vi:.4f} | OilIoU:{oil_iou:.4f}'
          f' | OilF1:{oil_f1:.4f} | Dice:{vd:.4f}')

    if vi > best_iou:
        best_iou = vi; counter = 0
        torch.save(seg_model.state_dict(), f'{MODEL_DIR}/best_model.pth')
        print('  ✅ Best model saved (mIoU)')
    else:
        counter += 1

    if score > best_score:
        best_score = score
        torch.save(seg_model.state_dict(), f'{MODEL_DIR}/best_model_score.pth')
        print('  ⭐ Best model saved (score)')

    if counter >= patience:
        print('Early stopping triggered.'); break

print('\n✅ Segmentation Training Complete')
print(f'Best mIoU: {best_iou:.4f}  |  Best Score: {best_score:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].plot(history['train_loss'],label='Train'); axes[0,0].plot(history['val_loss'],label='Val')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid()
axes[0,1].plot(history['iou'],linewidth=2,color='steelblue')
axes[0,1].set_title('Mean IoU'); axes[0,1].grid()
axes[0,2].plot(history['oil_iou'],linewidth=2,color='crimson',label='Oil IoU')
axes[0,2].plot(history['oil_f1'], linewidth=2,color='orange', label='Oil F1')
axes[0,2].set_title('Oil Class Metrics'); axes[0,2].legend(); axes[0,2].grid()
axes[1,0].plot(history['dice'],linewidth=2,color='green')
axes[1,0].set_title('Dice Score'); axes[1,0].grid()
axes[1,1].plot(history['lr'])
axes[1,1].set_title('Learning Rate (decoder)'); axes[1,1].grid()
if hard_examples:
    axes[1,2].hist(hard_examples, bins=30, color='tomato')
    axes[1,2].set_title('Hard Example Loss Distribution')
else:
    axes[1,2].set_visible(False)
plt.tight_layout(); plt.show()
pd.DataFrame(history).to_csv(f'{WORK_DIR}/history.csv', index=False)
print('✅ history.csv saved')

In [ ]:
seg_model.load_state_dict(torch.load(f'{MODEL_DIR}/best_model.pth', map_location=DEVICE))
seg_model.eval()
print(f'Best model loaded | mIoU: {best_iou:.4f}')

def tta_predict(model, image):
    model.eval()
    with torch.no_grad():
        preds = []
        preds.append(torch.softmax(model(image), dim=1))
        for dims in [[3],[2]]:
            f = torch.flip(image, dims)
            preds.append(torch.flip(torch.softmax(model(f),dim=1), dims))
        for k in [1,3]:
            r = torch.rot90(image, k, [2,3])
            preds.append(torch.rot90(torch.softmax(model(r),dim=1), -k, [2,3]))
    return torch.mean(torch.stack(preds), dim=0)

def refine_mask(mask):
    oil = (mask==1).astype(np.uint8)
    k   = np.ones((5,5),np.uint8)
    oil = cv2.morphologyEx(oil, cv2.MORPH_CLOSE, k)
    oil = cv2.morphologyEx(oil, cv2.MORPH_OPEN,  k)
    return oil

def remove_small(mask, min_area=200):
    n,labels,stats,_ = cv2.connectedComponentsWithStats(mask)
    cleaned = np.zeros_like(mask)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA] > min_area: cleaned[labels==i]=1
    return cleaned

def refine_prediction(mask):
    oil = refine_mask(mask)
    oil = remove_small(oil)
    final = mask.copy(); final[mask==1]=0; final[oil==1]=1
    return final

def predict(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    aug = valid_transform(image=rgb)
    x   = aug['image'].unsqueeze(0).to(DEVICE)
    probs = tta_predict(seg_model, x)
    pred  = torch.argmax(probs, dim=1).squeeze().cpu().numpy()
    pred  = refine_prediction(pred)
    return pred, probs.squeeze().cpu().numpy(), float(probs.max())

In [ ]:
test_files = get_valid_stems(TEST_IMG, TEST_MASK)
test_ds    = OilDataset(TEST_IMG, TEST_MASK, test_files, valid_transform)
test_loader= DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_ds)}')

all_preds = []; all_gts = []; inf_times = []
seg_model.eval()
with torch.no_grad():
    for img, mask in tqdm(test_loader, desc='Evaluating'):
        img = img.to(DEVICE)
        gt  = mask.squeeze(0).numpy()
        t0  = time.time()
        probs = tta_predict(seg_model, img)
        pred  = probs.argmax(dim=1).squeeze().cpu().numpy()
        pred  = refine_prediction(pred)
        inf_times.append(time.time()-t0)
        all_preds.append(pred.flatten()); all_gts.append(gt.flatten())

all_preds_flat = np.concatenate(all_preds)
all_gts_flat   = np.concatenate(all_gts)
eps = 1e-7

class_ious = []; class_dice = []
for c in range(NUM_CLASSES):
    inter = np.sum((all_preds_flat==c)&(all_gts_flat==c))
    union = np.sum((all_preds_flat==c)|(all_gts_flat==c))
    class_ious.append((inter+eps)/(union+eps))
    p = all_preds_flat==c; g = all_gts_flat==c
    class_dice.append((2*np.sum(p&g)+eps)/(np.sum(p)+np.sum(g)+eps))

mIoU      = float(np.mean(class_ious))
pixel_acc = float(np.mean(all_preds_flat==all_gts_flat))
op = all_preds_flat==1; og = all_gts_flat==1
prec = np.sum(op&og)/(np.sum(op)+eps)
rec  = np.sum(op&og)/(np.sum(og)+eps)
oil_f1 = 2*prec*rec/(prec+rec+eps)

print('='*60)
print(f'Pixel Accuracy : {pixel_acc:.4f}')
print(f'Mean IoU       : {mIoU:.4f}')
for c in range(NUM_CLASSES):
    print(f'  {CLASS_NAMES[c]:<12} IoU: {class_ious[c]:.4f}  Dice: {class_dice[c]:.4f}')
print(f'Oil Precision  : {prec:.4f}')
print(f'Oil Recall     : {rec:.4f}')
print(f'Oil F1         : {oil_f1:.4f}')
print(f'Inference Time : {np.mean(inf_times)*1000:.2f} ms')
print('='*60)

In [ ]:
cm      = confusion_matrix(all_gts_flat, all_preds_flat, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(np.float32)/(cm.sum(axis=1,keepdims=True)+1e-9)
fig,ax = plt.subplots(1,2,figsize=(14,6))
for a, mat, title, fmt_fn in [
    (ax[0], cm,      'Confusion Matrix',            lambda x: str(x)),
    (ax[1], cm_norm, 'Normalized Confusion Matrix', lambda x: f'{x:.2f}'),
]:
    im = a.imshow(mat, cmap='Blues' if a is ax[0] else 'Greens',
                  vmin=0, vmax=(None if a is ax[0] else 1))
    a.set_title(title)
    a.set_xticks(range(NUM_CLASSES)); a.set_xticklabels(CLASS_NAMES, rotation=20)
    a.set_yticks(range(NUM_CLASSES)); a.set_yticklabels(CLASS_NAMES)
    thresh = mat.max()/2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            a.text(j,i,fmt_fn(mat[i,j]),ha='center',va='center',
                   color='white' if mat[i,j]>thresh else 'black',fontsize=9)
    plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

In [ ]:
import json as _json
metrics = {
    'Pixel Accuracy': float(pixel_acc),
    'Mean IoU':       float(mIoU),
    'Oil IoU':        float(class_ious[1]),
    'Oil Dice':       float(class_dice[1]),
    'Oil Precision':  float(prec),
    'Oil Recall':     float(rec),
    'Oil F1':         float(oil_f1),
    'Inference ms':   float(np.mean(inf_times)*1000),
}
with open(f'{RESULT_DIR}/metrics.json','w') as f: _json.dump(metrics,f,indent=4)
with open(f'{MODEL_DIR}/class_colors.json','w') as f:
    _json.dump({str(k):list(v) for k,v in CLASS_COLORS.items()},f,indent=4)
torch.save(seg_model.state_dict(), f'{MODEL_DIR}/final_seg_model.pth')
torch.save(gen.state_dict(),        f'{MODEL_DIR}/final_generator.pth')
pd.DataFrame(history).to_csv(f'{WORK_DIR}/history.csv', index=False)
print('✅ All outputs saved to Google Drive')
print(f'   {WORK_DIR}')

In [ ]:
def draw_overlay(img, pred):
    ov = img.copy(); ov[pred==1]=[255,0,0]
    return cv2.addWeighted(img,0.65,ov,0.35,0)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
mean_n = np.array([0.485,0.456,0.406]); std_n = np.array([0.229,0.224,0.225])

for row, idx in enumerate(random.sample(range(len(test_ds)), min(4,len(test_ds)))):
    img_t, mask_t = test_ds[idx]
    img_np = np.clip(img_t.permute(1,2,0).numpy()*std_n + mean_n, 0, 1)
    img_u8 = (img_np*255).astype(np.uint8)
    gt_np  = mask_t.numpy()

    x = img_t.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = tta_predict(seg_model, x).squeeze().cpu().numpy()
    pred_np = probs.argmax(0); pred_np = refine_prediction(pred_np)

    axes[row,0].imshow(img_np);              axes[row,0].set_title('Image');      axes[row,0].axis('off')
    axes[row,1].imshow(mask_to_rgb(gt_np));  axes[row,1].set_title('GT Mask');    axes[row,1].axis('off')
    axes[row,2].imshow(mask_to_rgb(pred_np));axes[row,2].set_title('Pred Mask');  axes[row,2].axis('off')
    axes[row,3].imshow(draw_overlay(img_u8, pred_np)); axes[row,3].set_title('Overlay'); axes[row,3].axis('off')
    axes[row,4].imshow(probs[1], cmap='jet',vmin=0,vmax=1); axes[row,4].set_title('Oil Prob'); axes[row,4].axis('off')

plt.suptitle('Test Set Predictions', fontsize=16, y=1.01)
plt.tight_layout(); plt.show()